In [ ]:
# Problem: Time Series Forecasting of electricity consumption using LSTM (Deep Learning Intro)
# Dataset: https://www.kaggle.com/datasets/robikscube/hourly-energy-consumption


import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
import matplotlib.pyplot as plt


In [6]:
# Step 1: Load the data
data = pd.read_csv('AEP_hourly.csv')
print(data.info())
print(data.head())

# Convert the Datetime column to datetime object.
data['Datetime'] = pd.to_datetime(data['Datetime'])

# Set the Datetime as index.
data.set_index('Datetime', inplace=True)

# Check for missing values.
print(data.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121273 entries, 0 to 121272
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   Datetime  121273 non-null  object 
 1   AEP_MW    121273 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB
None
              Datetime   AEP_MW
0  2004-12-31 01:00:00  13478.0
1  2004-12-31 02:00:00  12865.0
2  2004-12-31 03:00:00  12577.0
3  2004-12-31 04:00:00  12517.0
4  2004-12-31 05:00:00  12670.0
AEP_MW    0
dtype: int64


In [7]:
# Step 2: Handle Missing Values (if any)

# If there would have been any missing values.
# First use iterpolation to predict missing values from neighbour data points.
data = data.interpolate()
# Drop the missing if left any.
data = data.dropna()

In [8]:
# Step 3: Normalize the Data
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data[['AEP_MW']])
# Check first five rows of scaled data.
print(data_scaled[:5])

[[0.24183939]
 [0.20379794]
 [0.18592528]
 [0.18220181]
 [0.19169666]]


In [9]:
# Step 4: Create Sequences for LSTM Model

# data: The entire time series data (in this case, the scaled electricity consumption data).
# time_steps: The number of previous time steps (hours) you want to use as input to predict the next time step.
def create_sequences(data, time_steps):
    sequences = [] # This list will store the input sequences (i.e., the previous 60 hours of electricity consumption).
    target = [] # This list will store the corresponding target values.

    for i in range(len(data) - time_steps):
        sequences.append(data[i: i + time_steps])
        target.append(data[i + time_steps])

    return np.array(sequences), np.array(target)

time_steps = 60
# Create sequences
X, y = create_sequences(data_scaled, time_steps)

# Check the shape of sequences
# print(X.shape, y.shape)
# Output: (121213, 60, 1) (121213, 1)


In [10]:
# Step 5: Split the data into training and validation datasets
train_size = int(len(X) * 0.8)
X_train, X_val = X[:train_size], X[train_size:]
y_train, y_val = y[:train_size], y[train_size:]

# Check the shape of train and validation dataset.
# print(X_train.shape, X_val.shape)
# print(y_train.shape, y_val.shape)


In [ ]:
# Step 6: Build and Train the LSTM Model
model = Sequential()

# Add an LSTM layer with 50 units
model.add(LSTM(50, return_sequences=False, input_shape=(X_train.shape[1], 1)))

# Add a Dense layer to output a single value.
model.add(Dense(1))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_val, y_val))

C:\Users\muhid\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
1110/3031 ━━━━━━━━━━━━━━━━━━━━ 1:07 35ms/step - loss: 0.0042